To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth your local device, follow [our guide](https://docs.unsloth.ai/get-started/install-and-update). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & [how to save it](#Save)


### News


Unsloth's [Docker image](https://hub.docker.com/r/unsloth/unsloth) is here! Start training with no setup & environment issues. [Read our Guide](https://docs.unsloth.ai/new/how-to-train-llms-with-unsloth-and-docker).

[gpt-oss RL](https://docs.unsloth.ai/new/gpt-oss-reinforcement-learning) is now supported with the fastest inference & lowest VRAM. Try our [new notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/gpt-oss-(20B)-GRPO.ipynb) which creates kernels!

Introducing [Vision](https://docs.unsloth.ai/new/vision-reinforcement-learning-vlm-rl) and [Standby](https://docs.unsloth.ai/basics/memory-efficient-rl) for RL! Train Qwen, Gemma etc. VLMs with GSPO - even faster with less VRAM.

Unsloth now supports Text-to-Speech (TTS) models. Read our [guide here](https://docs.unsloth.ai/basics/text-to-speech-tts-fine-tuning).

Visit our docs for all our [model uploads](https://docs.unsloth.ai/get-started/all-our-models) and [notebooks](https://docs.unsloth.ai/get-started/unsloth-notebooks).


### Installation

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

### Unsloth

`FastModel` supports loading nearly any model now! This includes Vision and Text models!

In [2]:
from unsloth import FastModel
import torch
max_seq_length = 1024
fourbit_models = [
    # 4bit dynamic quants for superior accuracy and low memory use
    "unsloth/gemma-3-1b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-4b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-27b-it-unsloth-bnb-4bit",

    # Other popular models!
    "unsloth/Llama-3.1-8B",
    "unsloth/Llama-3.2-3B",
    "unsloth/Llama-3.3-70B",
    "unsloth/mistral-7b-instruct-v0.3",
    "unsloth/Phi-4",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3-270m-it",
    max_seq_length = max_seq_length, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "hf_...", # use one if using gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.11.2: Fast Gemma3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.
Unsloth: Gemma3 does not support SDPA - switching to fast eager.


model.safetensors:   0%|          | 0.00/393M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

We now add LoRA adapters so we only need to update a small amount of parameters!

In [3]:
model = FastModel.get_peft_model(
    model,
    r = 8, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth: Making `model.base_model.model.model` require gradients


<a name="Data"></a>
### Data Prep
We now use the `Gemma-3` format for conversation style finetunes. We use [Thytu's ChessInstruct](https://huggingface.co/datasets/Thytu/ChessInstruct) dataset. Gemma-3 renders multi turn conversations like below:

```
<bos><start_of_turn>user
Hello!<end_of_turn>
<start_of_turn>model
Hey there!<end_of_turn>
```

We use our `get_chat_template` function to get the correct chat template. We support `zephyr, chatml, mistral, llama, alpaca, vicuna, vicuna_old, phi3, llama3, phi4, qwen2.5, gemma3` and more.

In [4]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma3",
)

In [39]:
dataset = load_dataset(
    "text",
    data_files="/content/finetune_data.txt",
    split="train[:50000]",
    keep_in_memory=True
)


Generating train split: 0 examples [00:00, ? examples/s]

We now use `convert_to_chatml` to try converting datasets to the correct format for finetuning purposes!

In [24]:
def convert_to_chatml(example):
    return {
        "conversations": [
            {"role": "system", "content": example["task"]},
            {"role": "user", "content": example["input"]},
            {"role": "assistant", "content": example["expected_output"]}
        ]
    }

dataset = dataset.map(
    convert_to_chatml
)

Map:   0%|          | 0/2871 [00:00<?, ? examples/s]

[{'content': '', 'role': 'system'}, {'content': 'Show the switches for gunzip.', 'role': 'user'}, {'content': '-c Output to stdout (act as zcat)\n-f Force: allow read from tty\n-k Keep input files (default is to remove)\n-t Test integrity', 'role': 'assistant'}]


Creating json from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

Saved chatml_dataset.jsonl — ready for fine-tuning.


Let's see how row 100 looks like!

In [27]:
print(ds[100])


{'task': '', 'input': 'What are the options for the rmmod command?', 'expected_output': '-f Force unload of a module\n-w Wait until the module is no longer used'}


We now have to apply the chat template for `Gemma3` onto the conversations, and save it to `text`.

In [40]:
# 1) Inspect dataset columns first (run this to check what's present)
print("Dataset columns:", dataset.column_names)
print("Number of examples:", len(dataset))
print("Example 0 keys/preview:", {k: dataset[0].get(k) for k in dataset.column_names})

# 2) Robust mapping function that supports several common shapes
def formatting_prompts_func(examples):
    texts = []

    # batched=True -> each value in examples is a list
    cols = list(examples.keys())

    # Case A: already-chatml "conversations" column (list-of-lists of {role,content})
    if "conversations" in cols:
        convos_batch = examples["conversations"]      # list of conversations
        for convo in convos_batch:
            # convo is expected to be e.g. [{"role":"system", "content":...}, ...]
            text = tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
            if isinstance(text, str) and text.startswith("<bos>"):
                text = text[len("<bos>"):]
            texts.append(text)

    # Case B: separate "input" (user) and "expected_output" (assistant)
    elif "input" in cols and "expected_output" in cols:
        inputs = examples["input"]
        outputs = examples["expected_output"]
        for u, o in zip(inputs, outputs):
            convo = [
                {"role":"system", "content": ""},          # empty or supply a system prompt if you want
                {"role":"user", "content": u},
                {"role":"assistant", "content": o}
            ]
            text = tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
            if isinstance(text, str) and text.startswith("<bos>"):
                text = text[len("<bos>"):]
            texts.append(text)

    # Case C: already flattened "text" column (just return it)
    elif "text" in cols:
        for t in examples["text"]:
            if isinstance(t, str) and t.startswith("<bos>"):
                t = t[len("<bos>"):]
            texts.append(t)

    # Unknown format -> raise clear error (won't silently fail)
    else:
        raise KeyError(
            "Dataset doesn't contain 'conversations', nor ('input' and 'expected_output'), nor 'text'.\n"
            f"Found columns: {cols}"
        )

    return {"text": texts}

# 3) Run the mapping
# Use num_proc=... only if your environment supports multiprocessing and tokenizer is picklable.
dataset = dataset.map(formatting_prompts_func, batched=True)
print("Mapping finished — new columns:", dataset.column_names)
print("Sample formatted text:", dataset[0]["text"][:400])


Dataset columns: ['text']
Number of examples: 42805
Example 0 keys/preview: {'text': '<start_of_turn>user'}


Map:   0%|          | 0/42805 [00:00<?, ? examples/s]

Mapping finished — new columns: ['text']
Sample formatted text: <start_of_turn>user


Let's see how the chat template did!


In [34]:
dataset[203]['text']

'Usage: /system/bin/linker [--list] PROGRAM [ARGS-FOR-PROGRAM...]'

In [46]:
# show columns & size
print("Columns:", dataset.column_names)
print("Examples:", len(dataset))
# peek one example (pick a valid index)
print("Example 0:", dataset[2])


Columns: ['text', 'input_ids', 'attention_mask', 'labels']
Examples: 42805
Example 0: {'text': '<end_of_turn>', 'input_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [42]:
# make tokenizer safe for padding/training
if getattr(tokenizer, "pad_token", None) is None:
    if getattr(tokenizer, "eos_token", None) is not None:
        tokenizer.pad_token = tokenizer.eos_token
    else:
        tokenizer.add_special_tokens({"pad_token": "<|endoftext|>"})
print("pad_token:", tokenizer.pad_token, "eos_token:", tokenizer.eos_token)


pad_token: <pad> eos_token: <end_of_turn>


In [43]:
def build_text_from_io(ex):
    # ex is a batch when batched=True
    texts = []
    for u, o in zip(ex["input"], ex["expected_output"]):
        # create a minimal conversation; you can set a system prompt if desired
        convo = [
            {"role":"system","content":""},
            {"role":"user","content": u},
            {"role":"assistant","content": o}
        ]
        # if you have a helper to render convo -> string, use it; otherwise simple join:
        txt = ""
        for turn in convo:
            role = "assistant" if turn["role"] == "assistant" else turn["role"]
            txt += f"<start_of_turn>{role}\n{turn['content']}\n<end_of_turn>\n"
        texts.append(txt)
    return {"text": texts}

# Only run if 'text' not already present
if "text" not in dataset.column_names:
    dataset = dataset.map(build_text_from_io, batched=True)
    print("Added 'text' column.")


In [44]:
max_length = 1024

def tokenize_for_sft(batch):
    texts = batch["text"]
    out = tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=max_length,
        return_attention_mask=True
    )
    out["labels"] = [ids.copy() for ids in out["input_ids"]]  # labels = input_ids for causal LM
    return out

# Remove other columns to keep dataset small on-disk
cols_to_remove = [c for c in dataset.column_names if c not in ("text",)]
dataset = dataset.map(tokenize_for_sft, batched=True, remove_columns=cols_to_remove)
print("Columns after tokenization:", dataset.column_names)
print("Sample shapes:", {k: len(dataset[0][k]) if isinstance(dataset[0].get(k), list) else type(dataset[0].get(k)) for k in dataset.column_names})


Map:   0%|          | 0/42805 [00:00<?, ? examples/s]

Columns after tokenization: ['text', 'input_ids', 'attention_mask', 'labels']
Sample shapes: {'text': <class 'str'>, 'input_ids': 1024, 'attention_mask': 1024, 'labels': 1024}


In [48]:
from torch.utils.data import DataLoader
import torch

def collate_fn(batch):
    # batch is a list of dicts
    return {
        "input_ids": torch.tensor([x["input_ids"] for x in batch], dtype=torch.long),
        "attention_mask": torch.tensor([x["attention_mask"] for x in batch], dtype=torch.long),
        "labels": torch.tensor([x["labels"] for x in batch], dtype=torch.long),
    }

# small DataLoader
dl = DataLoader(
    dataset.select(range(min(8, len(dataset)))),
    batch_size=2,
    shuffle=False,
    collate_fn=collate_fn
)

batch = next(iter(dl))
print("Batch keys:", batch.keys())
print("input_ids shape:", batch["input_ids"].shape)
print("labels shape:", batch["labels"].shape)


# quick model forward
device = next(model.parameters()).device
batch = {k: v.to(device) for k, v in batch.items()}
with torch.no_grad():
    out = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        labels=batch["labels"]
    )
print("Loss (smoke test):", out.loss.item())


Batch keys: dict_keys(['input_ids', 'attention_mask', 'labels'])
input_ids shape: torch.Size([2, 1024])
labels shape: torch.Size([2, 1024])
Loss (smoke test): 23.87229347229004


<a name="Train"></a>
### Train the model
Now let's train our model. We do 100 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

In [49]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None, # Can set up evaluation!
    args = SFTConfig(
    dataset_text_field="text",          # Which field in your dataset to use
    per_device_train_batch_size=8,      # Batch size per GPU/CPU
    gradient_accumulation_steps=1,      # Accumulates grads to simulate larger batch
    warmup_steps=5,                     # Learning rate warmup
    max_steps=100,                      # Total training steps (short for testing)
    learning_rate=5e-5,                 # Learning rate
    logging_steps=1,                    # Print every step
    optim="adamw_8bit",                 # Efficient 8-bit AdamW
    weight_decay=0.001,                 # Regularization
    lr_scheduler_type="linear",
    seed=3407,
    output_dir="outputs",
    report_to="none"
),
)

Unsloth: Switching to float32 training since model cannot work with float16


In [51]:
from datasets import Dataset

# Example: if your dataset is a HF Dataset
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])


We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs. This helps increase accuracy of finetunes!

In [52]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part="<start_of_turn>user\n",
    response_part="<start_of_turn>model\n",
)


Map (num_proc=6):   0%|          | 0/42805 [00:00<?, ? examples/s]

Let's verify masking the instruction part is done! Let's print the 100th row again.

In [54]:
for i in range(10):
    text = tokenizer.decode(trainer.train_dataset[i]["input_ids"], skip_special_tokens=True)
    print(f"{i}: {text}\n---")


0: user
---
1: Show the switches for gunzip.
---
2: 
---
3: model
---
4: -c Output to stdout (act as zcat)
---
5: -f Force: allow read from tty
---
6: -k Keep input files (default is to remove)
---
7: -t Test integrity
---
8: 
---
9: 
---


Now let's print the masked out example - you should see only the answer is present:

In [67]:
raw_dataset = load_dataset("json", data_files="/content/chatml_dataset.jsonl")["train"]


Generating train split: 0 examples [00:00, ? examples/s]

In [68]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.741 GB.
2.621 GB of memory reserved.


In [72]:
from datasets import load_dataset

# 1️⃣ Load your dataset
raw_dataset = load_dataset("json", data_files="/content/chatml_dataset.jsonl")["train"]

# 2️⃣ Inspect a sample to verify fields
print("Sample record:", raw_dataset[0])

# ✅ Updated tokenization function
def tokenize_for_sft(example):
    # Use processing_class directly
    tokenizer = trainer.processing_class

    # Get input and output text from your dataset
    input_text = example.get("input", "")
    output_text = example.get("expected_output", "")

    # Concatenate them for SFT
    full_text = input_text + "\n" + output_text

    tokenized = tokenizer(
        full_text,
        padding="max_length",
        truncation=True,
        max_length=1024
    )

    # For SFT, labels are same as input_ids
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

# Apply tokenization to the dataset
trainer.train_dataset = raw_dataset.map(tokenize_for_sft, batched=False)



Sample record: {'task': '', 'input': 'Show the switches for gunzip.', 'expected_output': '-c Output to stdout (act as zcat)\n-f Force: allow read from tty\n-k Keep input files (default is to remove)\n-t Test integrity', 'conversations': [{'content': '', 'role': 'system'}, {'content': 'Show the switches for gunzip.', 'role': 'user'}, {'content': '-c Output to stdout (act as zcat)\n-f Force: allow read from tty\n-k Keep input files (default is to remove)\n-t Test integrity', 'role': 'assistant'}]}


Map:   0%|          | 0/2871 [00:00<?, ? examples/s]

Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [73]:
# 6️⃣ Start training
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,871 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 1 x 1) = 8
 "-____-"     Trainable parameters = 1,898,496 of 269,996,672 (0.70% trained)


Step,Training Loss
1,16.997400
2,19.191700
3,18.045800
4,15.222500
5,11.672300
6,9.089400
7,7.379700
8,5.811600
9,4.593000
10,4.691300


Unsloth: Will smartly offload gradients to save VRAM!


In [74]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

232.1908 seconds used for training.
3.87 minutes used for training.
Peak reserved memory = 8.928 GB.
Peak reserved memory for training = 6.307 GB.
Peak reserved memory % of max memory = 60.566 %.
Peak reserved memory for training % of max memory = 42.785 %.


<a name="Inference"></a>
### Inference
Let's run the model via Unsloth native inference! According to the `Gemma-3` team, the recommended settings for inference are `temperature = 1.0, top_p = 0.95, top_k = 64`

In [78]:
from transformers import TextStreamer
import torch

# Pick an example index
idx = 10

# Convert input_ids back to text
input_ids = dataset[idx]["input_ids"]
prompt_text = tokenizer.decode(input_ids, skip_special_tokens=True)

# Optional: remove <bos> or other special tokens
prompt_text = prompt_text.lstrip("<bos>").strip()

# Tokenize for generation
inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")

# Generate with Gemma-3 recommended settings
streamer = TextStreamer(tokenizer, skip_prompt=True)
_ = model.generate(
    **inputs,
    max_new_tokens=125,
    temperature=1.0,
    top_p=0.95,
    top_k=64,
    streamer=streamer,
)


: I'm wondering if you know of any good resources for learning to code.

... I'm not sure I'm ready for the initial steps, but here's some general resources on learning to code:

What is it with learning to code? Learning to code is a broad, general-purpose, and multi-faceted approach to learning to code. It's about exploring by exploring, experimenting, and using various programming languages and platforms to learn new skills and develop new knowledge. It is about learning a language (like Python, Javascript, Swift, or Objective-C) through hands-on coding


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [79]:
model.save_pretrained("gemma-3")  # Local saving
tokenizer.save_pretrained("gemma-3")
# model.push_to_hub("your_name/gemma-3", token = "...") # Online saving
# tokenizer.push_to_hub("your_name/gemma-3", token = "...") # Online saving

('gemma-3/tokenizer_config.json',
 'gemma-3/special_tokens_map.json',
 'gemma-3/chat_template.jinja',
 'gemma-3/tokenizer.model',
 'gemma-3/added_tokens.json',
 'gemma-3/tokenizer.json')

In [81]:
import os

save_dir = "/content/drive/MyDrive/gemma-3"  # Change the path as you like
os.makedirs(save_dir, exist_ok=True)


In [82]:
# Save LoRA adapters + tokenizer
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)


('/content/drive/MyDrive/gemma-3/tokenizer_config.json',
 '/content/drive/MyDrive/gemma-3/special_tokens_map.json',
 '/content/drive/MyDrive/gemma-3/chat_template.jinja',
 '/content/drive/MyDrive/gemma-3/tokenizer.model',
 '/content/drive/MyDrive/gemma-3/added_tokens.json',
 '/content/drive/MyDrive/gemma-3/tokenizer.json')

In [85]:
import shutil

import os

drive_dir = "/content/drive/MyDrive/gemma-3"
os.makedirs(drive_dir, exist_ok=True)


shutil.copytree("/content/gemma-3", drive_dir, dirs_exist_ok=True)


'/content/drive/MyDrive/gemma-3'

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [87]:
if True:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "/content/drive/MyDrive/gemma-3",  # path to saved model
        max_seq_length = 2048,
        load_in_4bit = True,  # load in 4-bit for efficient inference
    )


==((====))==  Unsloth 2025.11.2: Fast Gemma3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.
Unsloth: Gemma3 does not support SDPA - switching to fast eager.


### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [89]:
# Path on Google Drive
save_path = "/content/drive/MyDrive/gemma-3-finetune-16bit"

# Merge LoRA adapters with base model and save as 16-bit
model.save_pretrained_merged(
    save_path,
    tokenizer,
    save_method="merged_16bit"  # ✅ merged float16
)

print(f"✅ Model saved to {save_path} in 16-bit format")


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `/content/drive/MyDrive/gemma-3-finetune-16bit`: 100%|██████████| 1/1 [00:06<00:00,  6.33s/it]


Successfully copied all 1 files from cache to `/content/drive/MyDrive/gemma-3-finetune-16bit`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:16<00:00, 16.71s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/gemma-3-finetune-16bit`
✅ Model saved to /content/drive/MyDrive/gemma-3-finetune-16bit in 16-bit format


### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now for all models! For now, you can convert easily to `Q8_0, F16 or BF16` precision. `Q4_K_M` for 4bit will come later!

In [ ]:
model.save_pretrained_gguf(
    "/content/drive/MyDrive/gemma-3-finetune-gguf",
    tokenizer,
    quantization_method="Q8_0",  # Currently the best option for Android/llama.cpp
)


Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### Your chat template has a BOS token. We shall remove it temporarily.


Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `/content/drive/MyDrive/gemma-3-finetune-gguf`: 100%|██████████| 1/1 [00:15<00:00, 15.08s/it]


Successfully copied all 1 files from cache to `/content/drive/MyDrive/gemma-3-finetune-gguf`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:27<00:00, 27.76s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/gemma-3-finetune-gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q8_0'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: All required system packages already installed!
Unsloth: Install llama.cpp and building - please wait 1 to 3 minutes
Unsloth: Cloning llama.cpp repository
Unsloth: Install GGUF and other packages


Likewise, if you want to instead push to GGUF to your Hugging Face account, set `if False` to `if True` and add your Hugging Face token and upload location!

In [ ]:
if False: # Change to True to upload GGUF
    model.push_to_hub_gguf(
        "gemma-3-finetune",
        tokenizer,
        quantization_method = "Q8_0", # Only Q8_0, BF16, F16 supported
        repo_id = "HF_ACCOUNT/gemma-finetune-gguf",
        token = "hf_...",
    )

Now, use the `gemma-3-finetune.gguf` file or `gemma-3-finetune-Q4_K_M.gguf` file in llama.cpp.

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other links:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
6. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://docs.unsloth.ai/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>

  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).
